# Notebook 4: Add Sentiment Labels to Kaiaulu

By now, you should have:
1. Sentiment labels in MySQL (7,122 GitHub comments labeled positive, negative, or neutral)
2. Comment data freshly downloaded from GitHub via Kaiaulu (e.g., file paths, commit SHAs, review IDs, timestamps)

Neither is complete on its own. The Gold Standard has polarity labels but no GitHub data. Kaiaulu's output has data but no sentiment labels. We'll query the labels from MySQL, INNER JOIN them against Kaiaulu's downloaded comment data on `comment_id`, and write the result back into Kaiaulu's directory so `sentiment_analysis.Rmd` can use it directly.

### Before you start

Two things need to be in place before running any cells:

| What | Where it comes from |
|---|---|
| MySQL database with `comment_sentiment` table | Output of Notebook 1 |
| Kaiaulu rawdata for your selected project | Output of running `vignettes/download_github_events.Rmd` and `vignettes/download_github_pull_request_comments.Rmd` in Kaiaulu |

The Kaiaulu vignettes write their output to `vignettes/rawdata/github/{owner}/{repo}/` inside your local Kaiaulu directory. That's where this notebook reads from.

If those CSVs are missing, go back and run the corresponding vignettes in Notebook 3 first.

### Step 1: Import dependencies

In [23]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

### Step 2: Configure Project

Set `OWNER` and `REPO` to match the project you ran the Kaiaulu vignettes for. Set `KAIAULU_REPO` to your local Kaiaulu directory. This is where the notebook will read Kaiaulu's downloaded CSVs from and where it will write the joined output. MySQL credentials should match what you used in Notebooks 1-3.

In [ ]:
# Configure these before running
OWNER = "ADD_OWNER_HERE"   # GitHub repo owner
REPO  = "ADD_REPO_HERE"   # GitHub repo name

# Path to your local Kaiaulu directory
KAIAULU_REPO = Path("PATH_TO/kaiaulu")

# Kaiaulu rawdata directory for this project
KAIAULU_DATA_DIR = KAIAULU_REPO / "vignettes" / "rawdata" / "github" / OWNER / REPO

# MySQL connection
MYSQL_HOST     = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT     = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_DB       = os.getenv("MYSQL_DB", "github")
MYSQL_USER     = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "ADD_PASSWORD_HERE")

### Step 3: Query sentiment labels from MySQL

Pull the sentiment labels for your project directly from the `comment_sentiment` table. We join through GHTorrent to filter down to just the comments belonging to `OWNER/REPO`, and grab a context columns.

In [37]:
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

commit_sql = """
SELECT
    cs.ID AS comment_id,
    cs.Polarity AS polarity,
    cs.Text AS text,
    cc.created_at AS created_at,
    u.login AS author_login,
    u_owner.login AS owner,
    p.name AS repo
FROM comment_sentiment cs
JOIN commit_comments cc ON cs.ID = cc.comment_id
JOIN commits c ON cc.commit_id = c.id
JOIN projects p ON c.project_id = p.id
JOIN users u ON cc.user_id = u.id
JOIN users u_owner ON p.owner_id = u_owner.id
WHERE LOWER(u_owner.login) = :owner
  AND LOWER(p.name) = :repo
"""

pr_sql = """
SELECT
    cs.ID AS comment_id,
    cs.Polarity AS polarity,
    cs.Text AS text,
    prc.created_at AS created_at,
    u.login AS author_login,
    u_owner.login AS owner,
    p.name AS repo
FROM comment_sentiment cs
JOIN pull_request_comments prc ON cs.ID = prc.comment_id
JOIN pull_requests pr ON prc.pull_request_id = pr.id
JOIN projects p ON pr.base_repo_id = p.id
JOIN users u ON prc.user_id = u.id
JOIN users u_owner ON p.owner_id = u_owner.id
WHERE LOWER(u_owner.login) = :owner
  AND LOWER(p.name) = :repo
"""

params = {"owner": OWNER.lower(), "repo": REPO.lower()}

with engine.connect() as con:
    commit_labels = pd.read_sql(text(commit_sql), con, params=params)
    pr_labels     = pd.read_sql(text(pr_sql),     con, params=params)

# Deduplicate 85 comment IDs that appear in both commit_comments and pull_request_comments GHTorrent tables
combined = pd.concat([commit_labels, pr_labels], ignore_index=True)
project_ctx = combined.drop_duplicates(subset="comment_id", keep="first").copy()

dupes_dropped = len(combined) - len(project_ctx)
print(f"Commit comment sentiment labels: {len(commit_labels)}")
print(f"PR inline sentiment labels: {len(pr_labels)}")
if dupes_dropped > 0:
    print(f"Duplicate IDs removed: {dupes_dropped} (appeared in both tables)")
print(f"Total sentiment labels for {REPO}: {len(project_ctx)}")
project_ctx.head()

Commit comment sentiment labels: 33
PR inline sentiment labels: 111
Total sentiment labels for cakephp: 144


,comment_id,polarity,text,created_at,author_login,owner,repo
0,3411503,neutral,This is causing the year to return always with...,2013-06-12 09:21:03,luksm,cakephp,cakephp
1,2245783,neutral,@Scottymeuk Read the associated ticket [#3283]...,2012-12-03 11:25:13,ADmad,cakephp,cakephp
2,1040482,neutral,"https://github.com/petteyg/code_check""",2012-03-04 07:12:51,josegonzalez,cakephp,cakephp
3,998908,positive,"I'm an idiot, I don't know how I missed that t...",2012-02-22 14:22:03,markstory,cakephp,cakephp
4,744111,negative,"Sorry, guys. Yes individually tests pass. I am...",2011-11-24 01:02:20,ceeram,cakephp,cakephp


### Step 4: Remap polarity labels to integers

The Gold Standard uses strings (`"positive"`, `"negative"`, `"neutral"`). Kaiaulu's `sentiment_analysis.Rmd` expects integers: `0` = neutral, `1` = positive, `2` = negative. We remap here so the output is ready to use directly.

In [38]:
polarity_map = {"neutral": 0, "positive": 1, "negative": 2}

if project_ctx["polarity"].dtype == object:
    project_ctx["polarity"] = project_ctx["polarity"].str.lower().map(polarity_map)

unmapped = project_ctx["polarity"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows could not be mapped. Check for unexpected polarity strings")
else:
    counts = project_ctx["polarity"].value_counts().rename({0: "neutral", 1: "positive", 2: "negative"})
    print("All polarity labels mapped successfully.")
    print(counts.to_string())

All polarity labels mapped successfully.
polarity
neutral     82
negative    45
positive    17


### Step 5: Load the Kaiaulu output CSVs

Read the two CSVs that Kaiaulu's vignettes wrote into the `rawdata/` directory.

In [39]:
commit_csv_path = KAIAULU_DATA_DIR / f"{REPO}_commit_comments.csv"
pr_csv_path     = KAIAULU_DATA_DIR / f"{REPO}_pr_inline_comments.csv"

kaiaulu_commit = pd.read_csv(commit_csv_path)
kaiaulu_pr     = pd.read_csv(pr_csv_path)

print(f"Kaiaulu commit comments: {len(kaiaulu_commit)} rows, columns: {list(kaiaulu_commit.columns)}")
print(f"Kaiaulu PR inline comments: {len(kaiaulu_pr)} rows, columns: {list(kaiaulu_pr.columns)}")

Kaiaulu commit comments: 1569 rows, columns: ['comment_id', 'commit_id', 'author_login', 'author_id', 'body', 'created_at', 'updated_at']
Kaiaulu PR inline comments: 6100 rows, columns: ['review_id', 'comment_id', 'html_url', 'created_at', 'updated_at', 'comment_user_login', 'author_association', 'file_path', 'start_line', 'line', 'original_start_line', 'original_line', 'position', 'diff_hunk', 'body', 'commit_id']


### Step 6: INNER JOIN - Commit Comments

Join the MySQL sentiment labels against Kaiaulu's commit comments on `comment_id`.

In [45]:
commit_joined = project_ctx.merge(
    kaiaulu_commit,
    on='comment_id',
    how='inner',
    suffixes=('_gold', '_kaiaulu')
)

commit_dropped = len(project_ctx) - len(commit_joined)
print(f"{REPO} rows in sentiment labels: {len(project_ctx)}")
print(f"Rows matched in Kaiaulu commit comments: {len(commit_joined)}")

print("\nJoined commit comments (first 5 rows):")
display(commit_joined.head())

out_path = KAIAULU_DATA_DIR / f"{REPO}_sentiment_commit_comments_joined.csv"
commit_joined.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

cakephp rows in sentiment labels: 144
Rows matched in Kaiaulu commit comments: 33

Joined commit comments (first 5 rows):


,comment_id,polarity,text,created_at_gold,author_login_gold,owner,repo,commit_id,author_login_kaiaulu,author_id,body,created_at_kaiaulu,updated_at
0,3411503,neutral,This is causing the year to return always with...,2013-06-12 09:21:03,luksm,cakephp,cakephp,fd72f894ad091bbe3c10314091ee3ab34769afa2,luksm,868687,This is causing the year to return always with...,2013-06-12T21:21:03Z,2013-06-12T21:21:03Z
1,2245783,neutral,@Scottymeuk Read the associated ticket [#3283]...,2012-12-03 11:25:13,ADmad,cakephp,cakephp,ea467e72d72e9eb7cd140816ee8d7abd900b2629,ADmad,142658,@Scottymeuk Read the associated ticket [#3283]...,2012-12-03T22:25:13Z,2012-12-03T22:25:13Z
2,1040482,neutral,"https://github.com/petteyg/code_check""",2012-03-04 07:12:51,josegonzalez,cakephp,cakephp,a6da7361494b85411f1b93ea589e58405a77524b,josegonzalez,65675,https://github.com/petteyg/code_check\n,2012-03-04T18:12:51Z,2012-03-04T18:12:51Z
3,998908,positive,"I'm an idiot, I don't know how I missed that t...",2012-02-22 14:22:03,markstory,cakephp,cakephp,89df484fc5a93fac7b01bdf086a395ecf284217d,markstory,24086,"I'm an idiot, I don't know how I missed that t...",2012-02-23T01:22:03Z,2012-02-23T01:22:03Z
4,744111,negative,"Sorry, guys. Yes individually tests pass. I am...",2011-11-24 01:02:20,ceeram,cakephp,cakephp,05940ae1ec703a23714ff815e4e2e19cd1c6b5b7,ceeram,111448,"Sorry, guys. Yes individually tests pass. I am...",2011-11-24T12:02:20Z,2011-11-24T12:02:20Z



Saved: /Users/sheilalimon/Desktop/github/kaiaulu-sentiment/vignettes/rawdata/github/cakephp/cakephp/cakephp_sentiment_commit_comments_joined.csv


### Step 7: INNER JOIN - PR inline comments

Same join as Step 6, but against Kaiaulu's PR inline comments.

In [44]:
pr_joined = project_ctx.merge(
    kaiaulu_pr,
    on='comment_id',
    how='inner',
    suffixes=('_gold', '_kaiaulu')
)

pr_dropped = len(project_ctx) - len(pr_joined)
print(f"{REPO} rows in sentiment labels: {len(project_ctx)}")
print(f"Rows matched in Kaiaulu PR inline comments: {len(pr_joined)}")

print("\nJoined PR inline comments (first 5 rows):")
display(pr_joined.head())

out_path = KAIAULU_DATA_DIR / f"{REPO}_sentiment_pr_inline_comments_joined.csv"
pr_joined.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

cakephp rows in sentiment labels: 144
Rows matched in Kaiaulu PR inline comments: 111

Joined PR inline comments (first 5 rows):


,comment_id,polarity,text,created_at_gold,author_login,owner,repo,review_id,html_url,created_at_kaiaulu,...,author_association,file_path,start_line,line,original_start_line,original_line,position,diff_hunk,body,commit_id
0,6044242,neutral,I had it implemented that way originally. The ...,2013-08-28 07:51:55,markstory,cakephp,cakephp,NaN,https://github.com/cakephp/cakephp/pull/1568#d...,2013-08-28T19:51:55Z,...,MEMBER,lib/Cake/Utility/Security.php,NaN,NaN,NaN,NaN,1,"@@ -289,4 +289,69 @@ protected static function...",I had it implemented that way originally. The ...,13b870d7e183375822eea4ffd66aaacaeec760ff
1,5949747,neutral,This block of code is repeated 3 times in Hash...,2013-08-23 01:02:48,markstory,cakephp,cakephp,NaN,https://github.com/cakephp/cakephp/pull/1549#d...,2013-08-23T13:02:48Z,...,MEMBER,lib/Cake/Utility/Hash.php,NaN,149.0,NaN,NaN,30,"@@ -222,16 +222,36 @@ protected static functio...",This block of code is repeated 3 times in Hash...,a0014e7a303067bb9c36d438de5a70fe819d22a7
2,4288367,neutral,"This looks good, but makes me think we should ...",2013-05-18 04:31:19,markstory,cakephp,cakephp,NaN,https://github.com/cakephp/cakephp/pull/1275#d...,2013-05-18T16:31:19Z,...,MEMBER,lib/Cake/Controller/Component/Auth/BlowfishPas...,NaN,44.0,NaN,NaN,44,"@@ -0,0 +1,58 @@\n+<?php\n+/**\n+ * PHP 5\n+ *...","This looks good, but makes me think we should ...",dd2892ad8d0e3a0b09990b0a9ef26c320f1901fa
3,4288664,neutral,"Hmm, my thinking was all password hasher class...",2013-05-18 07:00:03,ADmad,cakephp,cakephp,NaN,https://github.com/cakephp/cakephp/pull/1275#d...,2013-05-18T19:00:03Z,...,MEMBER,lib/Cake/Controller/Component/Auth/BlowfishPas...,NaN,NaN,NaN,NaN,1,"@@ -0,0 +1,58 @@\n+<?php\n+/**\n+ * PHP 5\n+ *...","Hmm, my thinking was all password hasher class...",dd2892ad8d0e3a0b09990b0a9ef26c320f1901fa
4,3122764,negative,"I totally missed that, my bad. I'll get that f...",2013-02-22 10:12:40,markstory,cakephp,cakephp,NaN,https://github.com/cakephp/cakephp/pull/1154#d...,2013-02-22T21:12:40Z,...,MEMBER,lib/Cake/Utility/ViewVarsTrait.php,NaN,NaN,NaN,NaN,1,"@@ -0,0 +1,55 @@\n+<?php\n+/**\n+ * CakePHP(tm...","I totally missed that, my bad. I'll get that f...",955889c6c731a56f9cbe6f572cea4594fd887d3a



Saved: /Users/sheilalimon/Desktop/github/kaiaulu-sentiment/vignettes/rawdata/github/cakephp/cakephp/cakephp_sentiment_pr_inline_comments_joined.csv


### You're done!

The joined CSVs are saved into Kaiaulu's `vignettes/rawdata/` directory, right next to the raw downloads they came from:

```
vignettes/rawdata/github/{owner}/{repo}/
  {repo}_sentiment_commit_comments_joined.csv
  {repo}_sentiment_pr_inline_comments_joined.csv
```

Each row has polarity as an integer (`0` = neutral, `1` = positive, `2` = negative) and all the data Kaiaulu's `sentiment_analysis.Rmd` needs (comment body, author, timestamp, file path, commit SHA).

**To run for a different project:** update `OWNER` and `REPO` in Step 2, make sure the Kaiaulu vignettes have run for that project, and re-run Steps 3-7.